# IHDP benchmark with known ground truth

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [1]:
import numpy as np
import urllib.request

urllib.request.urlretrieve(
    "https://www.fredjo.com/files/ihdp_npci_1-100.train.npz", "ihdp_train.npz")
urllib.request.urlretrieve(
    "https://www.fredjo.com/files/ihdp_npci_1-100.test.npz", "ihdp_test.npz")

train = np.load("ihdp_train.npz")
test = np.load("ihdp_test.npz")

print("Keys:", train.files)
print("x shape (covariates):", train['x'].shape)
print("t shape (treatment):", train['t'].shape)
print("yf shape (factual outcome):", train['yf'].shape)
print("mu0, mu1 shape (true potential outcomes):", train['mu0'].shape, train['mu1'].shape)

x0, t0, yf0 = train['x'][:, :, 0], train['t'][:, 0], train['yf'][:, 0]
mu0_0, mu1_0 = train['mu0'][:, 0], train['mu1'][:, 0]

true_ite = mu1_0 - mu0_0
true_ate = true_ite.mean()
print(f"\nRealization 0 — true ATE: {true_ate:.4f}")
print(f"Treated: {t0.sum():.0f}, Control: {(1-t0).sum():.0f}")
print(f"True ITE range: [{true_ite.min():.2f}, {true_ite.max():.2f}]")

import hashlib, os
for f in ('ihdp_train.npz', 'ihdp_test.npz'):
    with open(f, 'rb') as fh:
        print(f"{f:<18} {os.path.getsize(f):>10,} bytes   sha256[:8] = "
              f"{hashlib.sha256(fh.read()).hexdigest()[:8]}")

assert train['x'].shape == (672, 25, 100), f"IHDP shape drift: {train['x'].shape}"
assert t0.sum() == 123 and (1 - t0).sum() == 549, "treated/control split drift"
print("\nIHDP verified: 672 units, 123 treated / 549 control, 100 realisations.")

Keys: ['ate', 'mu1', 'mu0', 'yadd', 'yf', 'ycf', 't', 'x', 'ymul']
x shape (covariates): (672, 25, 100)
t shape (treatment): (672, 100)
yf shape (factual outcome): (672, 100)
mu0, mu1 shape (true potential outcomes): (672, 100) (672, 100)

Realization 0 — true ATE: 4.0116
Treated: 123, Control: 549
True ITE range: [-1.87, 4.67]
ihdp_train.npz     16,129,570 bytes   sha256[:8] = 750697c7
ihdp_test.npz       1,801,570 bytes   sha256[:8] = a70a8acb

IHDP verified: 672 units, 123 treated / 549 control, 100 realisations.


In [2]:
!pip install econml

In [3]:
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

x, t, yf = train['x'][:, :, 0], train['t'][:, 0], train['yf'][:, 0]
mu0, mu1 = train['mu0'][:, 0], train['mu1'][:, 0]
true_ite = mu1 - mu0
true_ate = true_ite.mean()

naive_ate = yf[t == 1].mean() - yf[t == 0].mean()

ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                  model_t=GradientBoostingRegressor(random_state=42),
                  cv=5, random_state=42)
ldml.fit(yf, t, X=x)
ldml_ate = ldml.ate(x)

cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=GradientBoostingRegressor(random_state=42),
                      n_estimators=1000, min_samples_leaf=10, honest=True,
                      cv=5, random_state=42)
cf.fit(yf, t, X=x)
cf_ate = cf.ate(x)
cf_ite = cf.effect(x)

pehe = np.sqrt(np.mean((cf_ite - true_ite) ** 2))
ate_err_naive = abs(naive_ate - true_ate)
ate_err_ldml = abs(ldml_ate - true_ate)
ate_err_cf = abs(cf_ate - true_ate)

print(f"True ATE:              {true_ate:.4f}")
print(f"Naive diff-in-means:   {naive_ate:.4f}  (abs error: {ate_err_naive:.4f})")
print(f"Linear DML:            {ldml_ate:.4f}  (abs error: {ate_err_ldml:.4f})")
print(f"CausalForestDML:       {cf_ate:.4f}  (abs error: {ate_err_cf:.4f})")
print(f"\nPEHE (sqrt mean squared ITE error): {pehe:.4f}")

True ATE:              4.0116
Naive diff-in-means:   4.0047  (abs error: 0.0069)
Linear DML:            3.8795  (abs error: 0.1320)
CausalForestDML:       3.8681  (abs error: 0.1435)

PEHE (sqrt mean squared ITE error): 0.6174


In [6]:
n_reps = 100
results = {'naive_ate_err': [], 'ldml_ate_err': [], 'cf_ate_err': [],
           'ldml_pehe': [], 'cf_pehe': []}

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    naive_ate = yf[t == 1].mean() - yf[t == 0].mean()

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=GradientBoostingRegressor(random_state=42),
                      cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)

    cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                          model_t=GradientBoostingRegressor(random_state=42),
                          n_estimators=1000, min_samples_leaf=10, honest=True,
                          cv=5, random_state=42)
    cf.fit(yf, t, X=x)
    cf_ite = cf.effect(x)

    results['naive_ate_err'].append(abs(naive_ate - true_ate))
    results['ldml_ate_err'].append(abs(ldml_ite.mean() - true_ate))
    results['cf_ate_err'].append(abs(cf_ite.mean() - true_ate))
    results['ldml_pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))
    results['cf_pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 10 == 0:
        print(f"...completed realization {i}")

results_arr = {k: np.array(v) for k, v in results.items()}
print("\n--- Summary across 100 realizations ---")
for k, v in results_arr.items():
    print(f"{k}: mean={v.mean():.4f} ± {v.std():.4f}, median={np.median(v):.4f}")

outlier_idx = np.where(results_arr['cf_pehe'] > 3 * np.median(results_arr['cf_pehe']))[0]
print(f"\nRealizations with CF PEHE > 3x median (heavy-tailed, likely exponential-surface draws): {outlier_idx.tolist()}")

cf_wins_ate = (results_arr['cf_ate_err'] < results_arr['ldml_ate_err']).sum()
cf_wins_pehe = (results_arr['cf_pehe'] < results_arr['ldml_pehe']).sum()
print(f"CF beats LDML on ATE error in {cf_wins_ate}/100 realizations")
print(f"CF beats LDML on PEHE in {cf_wins_pehe}/100 realizations")

...completed realization 0
...completed realization 10
...completed realization 20
...completed realization 30
...completed realization 40
...completed realization 50
...completed realization 60
...completed realization 70
...completed realization 80
...completed realization 90

--- Summary across 100 realizations ---
naive_ate_err: mean=0.2856 ± 0.3841, median=0.1505
ldml_ate_err: mean=0.4651 ± 0.6850, median=0.3046
cf_ate_err: mean=0.6985 ± 1.1051, median=0.3781
ldml_pehe: mean=2.6855 ± 3.7819, median=1.2615
cf_pehe: mean=3.9215 ± 5.7776, median=1.8165

Realizations with CF PEHE > 3x median (heavy-tailed, likely exponential-surface draws): [8, 9, 12, 20, 25, 27, 33, 36, 38, 52, 59, 67, 70, 80, 81, 83, 84, 85, 92, 97]
CF beats LDML on ATE error in 39/100 realizations
CF beats LDML on PEHE in 26/100 realizations


In [7]:
from sklearn.linear_model import LogisticRegressionCV

n_reps = 100
configs = {
    'leaf10': dict(min_samples_leaf=10, n_estimators=1000),
    'leaf20': dict(min_samples_leaf=20, n_estimators=1000),
    'leaf30': dict(min_samples_leaf=30, n_estimators=1000),
}

all_results = {name: {'ate_err': [], 'pehe': []} for name in configs}
ldml_results = {'ate_err': [], 'pehe': []}
naive_results = []

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    naive_results.append(abs((yf[t==1].mean() - yf[t==0].mean()) - true_ate))

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)
    ldml_results['ate_err'].append(abs(ldml_ite.mean() - true_ate))
    ldml_results['pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    for name, params in configs.items():
        cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                              model_t=LogisticRegressionCV(cv=5, random_state=42),
                              discrete_treatment=True, honest=True, cv=5,
                              random_state=42, **params)
        cf.fit(yf, t, X=x)
        cf_ite = cf.effect(x)
        all_results[name]['ate_err'].append(abs(cf_ite.mean() - true_ate))
        all_results[name]['pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 20 == 0:
        print(f"...completed realization {i}")

print(f"\nNaive: ate_err mean={np.mean(naive_results):.4f}, median={np.median(naive_results):.4f}")
print(f"LDML (discrete_treatment=True): ate_err mean={np.mean(ldml_results['ate_err']):.4f}, "
      f"median={np.median(ldml_results['ate_err']):.4f}, "
      f"pehe mean={np.mean(ldml_results['pehe']):.4f}, median={np.median(ldml_results['pehe']):.4f}")
for name in configs:
    a, p = all_results[name]['ate_err'], all_results[name]['pehe']
    print(f"CF {name}: ate_err mean={np.mean(a):.4f}, median={np.median(a):.4f}, "
          f"pehe mean={np.mean(p):.4f}, median={np.median(p):.4f}")

/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 0


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 20


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...

...completed realization 60


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 80


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 


Naive: ate_err mean=0.2856, median=0.1505
LDML (discrete_treatment=True): ate_err mean=0.5635, median=0.4684, pehe mean=2.7737, median=1.4300
CF leaf10: ate_err mean=0.6354, median=0.3452, pehe mean=3.8760, median=1.8206
CF leaf20: ate_err mean=0.6692, median=0.3594, pehe mean=4.2169, median=1.9471
CF leaf30: ate_err mean=0.6843, median=0.3574, pehe mean=4.4544, median=2.0571


In [11]:
import numpy as np, pandas as pd

_naive     = np.asarray(naive_results, dtype=float)
_ldml_ate  = np.asarray(ldml_results['ate_err'], dtype=float)
_ldml_pehe = np.asarray(ldml_results['pehe'], dtype=float)
_cf_ate    = np.asarray(all_results['leaf10']['ate_err'], dtype=float)
_cf_pehe   = np.asarray(all_results['leaf10']['pehe'], dtype=float)

for _label, _got, _pub in [('naive ATE err', _naive.mean(), 0.2856),
                           ('LDML ATE err',  _ldml_ate.mean(), 0.5635),
                           ('CF ATE err',    _cf_ate.mean(), 0.6354),
                           ('LDML PEHE',     _ldml_pehe.mean(), 2.7737),
                           ('CF PEHE',       _cf_pehe.mean(), 3.8760)]:
    assert abs(_got - _pub) < 5e-4, f"{_label}: {_got!r} != published {_pub}"

print("IHDP ATE RECOVERY ERROR — FULL PRECISION, ALL THREE ESTIMATORS")
print(f"{_naive.size} realisations · min_samples_leaf=10 · n_estimators=1000")
print("discrete_treatment=True · model_t=LogisticRegressionCV")

rows = []
for name, v in [('naive difference-in-means', _naive),
                ('Linear DML', _ldml_ate),
                ('CausalForestDML (leaf10)', _cf_ate)]:
    rows.append({'estimator': name, 'mean_abs_ATE_error': v.mean(),
                 'sd_ddof1': v.std(ddof=1), 'median': np.median(v),
                 'min': v.min(), 'max': v.max(), 'n': v.size})
    print(f"{name}")
    print(f"    mean   = {v.mean()!r}")
    print(f"    median = {np.median(v)!r}")
    print(f"    range  = [{v.min()!r}, {v.max()!r}]\n")

summary = pd.DataFrame(rows)
summary.to_csv(f'{BASE}/ihdp_estimator_comparison.csv', index=False)

print("PEHE (individual-level effect recovery)")
for name, v in [('Linear DML', _ldml_pehe), ('CausalForestDML', _cf_pehe)]:
    print(f"  {name:<22} mean = {v.mean()!r}")
    print(f"  {'':<22} sd   = {v.std(ddof=1)!r}      <-- also withheld")
    print(f"  {'':<22} med  = {np.median(v)!r}")
print("  naive difference-in-means   NOT DEFINED")
print("\n  The naive estimator returns one population number and no per-unit")
print("  estimates, so PEHE cannot be computed for it. That is the substantive")
print("  cannot enter the other -- which is the axis this study is about.")
print("\n  For R10: IHDP Setting B is semi-synthetic with known treated/control")
print("  overlap, so difference-in-means here is not the naive estimator it would")
print("  be on observational field data. Its ATE recovery is not evidence that")
print("  adjustment is unnecessary in general.")

_w_ate  = int((_cf_ate  < _ldml_ate ).sum())
_w_pehe = int((_cf_pehe < _ldml_pehe).sum())
assert (_w_ate, _w_pehe) == (69, 30), "win counts diverge -- investigate"
for _name in ('leaf10', 'leaf20', 'leaf30'):
    _a = np.asarray(all_results[_name]['ate_err'], dtype=float)
    _p = np.asarray(all_results[_name]['pehe'], dtype=float)
    print(f"  {_name}: ATE err {_a.mean():.4f} +/- {_a.std(ddof=1):.4f}   "
          f"PEHE {_p.mean():.4f} +/- {_p.std(ddof=1):.4f}")

print("Saved: ihdp_estimator_comparison.csv")


IHDP ATE RECOVERY ERROR — FULL PRECISION, ALL THREE ESTIMATORS
100 realisations · min_samples_leaf=10 · n_estimators=1000
discrete_treatment=True · model_t=LogisticRegressionCV

naive difference-in-means
    mean   = np.float64(0.28559182497003266)
    median = np.float64(0.15049924942610238)
    range  = [np.float64(0.0005718516734143364), np.float64(1.9345192456014875)]

Linear DML
    mean   = np.float64(0.5634733006467171)
    median = np.float64(0.4683545131495843)
    range  = [np.float64(0.03831034730961047), np.float64(2.9786600279567503)]

CausalForestDML (leaf10)
    mean   = np.float64(0.635415565605051)
    median = np.float64(0.34522070046109965)
    range  = [np.float64(0.002213581004605558), np.float64(5.460420197248652)]



In [12]:
import os
import numpy as np, pandas as pd

def f(x):
    """Full precision without the np.float64(...) wrapper, as a plain float."""
    return repr(float(x))

rows = [{'estimator': n, 'mean_abs_ATE_error': v.mean(), 'sd_ddof1': v.std(ddof=1),
         'median': np.median(v), 'min': v.min(), 'max': v.max(), 'n': v.size}
        for n, v in [('naive difference-in-means', _naive),
                     ('Linear DML', _ldml_ate),
                     ('CausalForestDML (leaf10)', _cf_ate)]]
summary = pd.DataFrame(rows)
summary.to_csv(f'{BASE}/ihdp_estimator_comparison.csv', index=False)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.9f}"))

print("PEHE (individual-level effect recovery)")
pehe_rows = []
for name, v in [('Linear DML', _ldml_pehe), ('CausalForestDML', _cf_pehe)]:
    pehe_rows.append({'estimator': name, 'pehe_mean': v.mean(), 'pehe_sd': v.std(ddof=1),
                      'pehe_median': np.median(v), 'min': v.min(), 'max': v.max()})
    print(f"  {name:<22} mean = {f(v.mean())}")
    print(f"  {'':<22} sd   = {f(v.std(ddof=1))}      <-- also withheld")
    print(f"  {'':<22} med  = {f(np.median(v))}")
    print(f"  {'':<22} max  = {f(v.max())}")
print("  naive difference-in-means   NOT DEFINED")
pd.DataFrame(pehe_rows).to_csv(f'{BASE}/ihdp_pehe_comparison.csv', index=False)

print("\n  The naive estimator returns one population number and no per-unit")
print("  estimates, so PEHE cannot be computed for it. That is the substantive")
print("  cannot enter the other -- which is the axis this study is about.")
print("\n  For R10: IHDP Setting B is semi-synthetic with known treated/control")
print("  overlap, so difference-in-means here is not the naive estimator it would")
print("  be on observational field data.")

_w_ate = int((_cf_ate < _ldml_ate).sum())
_w_pehe = int((_cf_pehe < _ldml_pehe).sum())
assert (_w_ate, _w_pehe) == (69, 30), "win counts diverge -- investigate"
print("MEAN vs MEDIAN — the comparison inverts")
print(f"  ATE error, mean   : CF {f(_cf_ate.mean())}  vs  LDML {f(_ldml_ate.mean())}")
print(f"                      CF is {100*(_cf_ate.mean()/_ldml_ate.mean()-1):+.1f}% -- WORSE")
print(f"  ATE error, median : CF {f(np.median(_cf_ate))}  vs  LDML {f(np.median(_ldml_ate))}")
print(f"                      CF is {100*(np.median(_cf_ate)/np.median(_ldml_ate)-1):+.1f}% -- BETTER")
print(f"\n  CF sd / LDML sd     = {_cf_ate.std(ddof=1)/_ldml_ate.std(ddof=1):.2f}x")
print(f"  CF sd / CF mean     = {_cf_ate.std(ddof=1)/_cf_ate.mean():.2f}")
print(f"  CF max / LDML max   = {_cf_ate.max()/_ldml_ate.max():.2f}x")
print("\n  Heavy right tail. CF is more accurate on typical realisations and")
print("  occasionally catastrophic, which is why it wins 69/100 while posting a")
print("  conclusion. Report mean, sd AND median for every row, and say which")
print("  one the claim rests on.")

leaf_rows = []
for _name in ('leaf10', 'leaf20', 'leaf30'):
    _a = np.asarray(all_results[_name]['ate_err'], dtype=float)
    _p = np.asarray(all_results[_name]['pehe'], dtype=float)
    leaf_rows.append({'config': _name, 'ate_err_mean': _a.mean(), 'ate_err_sd': _a.std(ddof=1),
                      'pehe_mean': _p.mean(), 'pehe_sd': _p.std(ddof=1),
                      'pehe_median': np.median(_p)})
    print(f"  {_name}: ATE err {_a.mean():.4f} +/- {_a.std(ddof=1):.4f}   "
          f"PEHE {_p.mean():.4f} +/- {_p.std(ddof=1):.4f}   med {np.median(_p):.4f}")
pd.DataFrame(leaf_rows).to_csv(f'{BASE}/ihdp_leaf_sweep.csv', index=False)

import sys, sklearn, econml
print(f"environment: python {sys.version.split()[0]} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · sklearn {sklearn.__version__} · econml {econml.__version__}")
print("drift; it was a config difference. But versions DO differ across sessions --")

print(f"Saved to {BASE}: ihdp_estimator_comparison.csv, ihdp_pehe_comparison.csv,")
print("ihdp_leaf_sweep.csv")

In [13]:
from sklearn.linear_model import LassoCV, LogisticRegressionCV

ldml_results2 = {'ate_err': [], 'pehe': []}
cf_results2 = {'ate_err': [], 'pehe': []}

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    ldml = LinearDML(model_y=LassoCV(cv=5, random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)
    ldml_results2['ate_err'].append(abs(ldml_ite.mean() - true_ate))
    ldml_results2['pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    cf = CausalForestDML(model_y=LassoCV(cv=5, random_state=42),
                          model_t=LogisticRegressionCV(cv=5, random_state=42),
                          discrete_treatment=True, honest=True, cv=5,
                          min_samples_leaf=10, n_estimators=1000, random_state=42)
    cf.fit(yf, t, X=x)
    cf_ite = cf.effect(x)
    cf_results2['ate_err'].append(abs(cf_ite.mean() - true_ate))
    cf_results2['pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 20 == 0:
        print(f"...completed realization {i}")

print(f"\nLDML (Lasso/Logistic nuisance): ate_err mean={np.mean(ldml_results2['ate_err']):.4f}, "
      f"median={np.median(ldml_results2['ate_err']):.4f}, "
      f"pehe mean={np.mean(ldml_results2['pehe']):.4f}, median={np.median(ldml_results2['pehe']):.4f}")
print(f"CF (Lasso/Logistic nuisance): ate_err mean={np.mean(cf_results2['ate_err']):.4f}, "
      f"median={np.median(cf_results2['ate_err']):.4f}, "
      f"pehe mean={np.mean(cf_results2['pehe']):.4f}, median={np.median(cf_results2['pehe']):.4f}")

/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 0


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 20


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...

...completed realization 60


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 80


/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV


LDML (Lasso/Logistic nuisance): ate_err mean=0.6580, median=0.5259, pehe mean=2.5909, median=1.2863
CF (Lasso/Logistic nuisance): ate_err mean=0.7067, median=0.3889, pehe mean=3.8424, median=1.7430


In [14]:
cf_ate_arr, cf_pehe_arr, ldml_ate_arr, ldml_pehe_arr = [], [], [], []

for i in range(100):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x); ldml_ite = ldml.effect(x)
    ldml_ate_arr.append(abs(ldml_ite.mean() - true_ate))
    ldml_pehe_arr.append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                          model_t=LogisticRegressionCV(cv=5, random_state=42),
                          discrete_treatment=True, honest=True, cv=5,
                          min_samples_leaf=10, n_estimators=1000, random_state=42)
    cf.fit(yf, t, X=x); cf_ite = cf.effect(x)
    cf_ate_arr.append(abs(cf_ite.mean() - true_ate))
    cf_pehe_arr.append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

cf_ate_arr, cf_pehe_arr = np.array(cf_ate_arr), np.array(cf_pehe_arr)
ldml_ate_arr, ldml_pehe_arr = np.array(ldml_ate_arr), np.array(ldml_pehe_arr)
print(f"CF beats LDML on ATE error: {(cf_ate_arr < ldml_ate_arr).sum()}/100")
print(f"CF beats LDML on PEHE: {(cf_pehe_arr < ldml_pehe_arr).sum()}/100")

/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.13/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

CF beats LDML on ATE error: 69/100
CF beats LDML on PEHE: 30/100


In [15]:
import sys, time
import numpy as np
import pandas as pd
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegressionCV

import econml, sklearn
print("econml:", econml.__version__, "| sklearn:", sklearn.__version__)

N_REPS     = 100
LEAF_GRID  = [2, 3, 5, 10, 20, 30]
N_TREES    = 1000
SEED       = 42

assert 'train' in dir(), "run the cell that loads `train` first"

def make_cf(leaf, seed=SEED):
    return CausalForestDML(
        model_y=GradientBoostingRegressor(random_state=seed),
        model_t=LogisticRegressionCV(random_state=seed),

        discrete_treatment=True, honest=True,
        n_estimators=N_TREES, min_samples_leaf=leaf, cv=5, random_state=seed)

def make_ldml(seed=SEED):
    return LinearDML(
        model_y=GradientBoostingRegressor(random_state=seed),
        model_t=LogisticRegressionCV(random_state=seed),
        discrete_treatment=True, cv=5, random_state=seed)

print("PART A — LEAF SWEEP, EXTENDED BELOW 10")

per_real, summary = [], []
t_start = time.time()

for leaf in LEAF_GRID:
    print(f"\n--- min_samples_leaf = {leaf} ---")
    a_cf, a_ld, a_nv, p_cf, p_ld, wins = [], [], [], [], [], 0
    for i in range(N_REPS):
        x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
        true_ite = train['mu1'][:, i] - train['mu0'][:, i]
        true_ate = true_ite.mean()

        naive = yf[t == 1].mean() - yf[t == 0].mean()

        cf = make_cf(leaf); cf.fit(yf, t, X=x); cf_ite = cf.effect(x)
        ld = make_ldml();   ld.fit(yf, t, X=x); ld_ite = ld.effect(x)

        pehe_cf = np.sqrt(np.mean((cf_ite - true_ite) ** 2))
        pehe_ld = np.sqrt(np.mean((ld_ite - true_ite) ** 2))
        wins += pehe_cf < pehe_ld

        a_nv.append(abs(naive - true_ate))
        a_cf.append(abs(cf_ite.mean() - true_ate))
        a_ld.append(abs(ld_ite.mean() - true_ate))
        p_cf.append(pehe_cf); p_ld.append(pehe_ld)
        per_real.append({'leaf': leaf, 'realisation': i, 'pehe_cf': pehe_cf,
                         'pehe_ldml': pehe_ld, 'ate_err_cf': a_cf[-1],
                         'ate_err_ldml': a_ld[-1], 'ate_err_naive': a_nv[-1],
                         'true_ate': true_ate})
        if (i + 1) % 25 == 0:
            print(f"    {i+1}/{N_REPS}  [{time.time()-t_start:.0f}s]")

    summary.append({
        'min_samples_leaf': leaf,
        'ATEerr_naive_mean': np.mean(a_nv), 'ATEerr_naive_sd': np.std(a_nv, ddof=1),
        'ATEerr_LDML_mean': np.mean(a_ld),  'ATEerr_LDML_sd': np.std(a_ld, ddof=1),
        'ATEerr_CF_mean': np.mean(a_cf),    'ATEerr_CF_sd': np.std(a_cf, ddof=1),
        'ATEerr_CF_median': np.median(a_cf),
        'PEHE_LDML_mean': np.mean(p_ld), 'PEHE_LDML_median': np.median(p_ld),
        'PEHE_CF_mean': np.mean(p_cf),   'PEHE_CF_median': np.median(p_cf),
        'CF_PEHE_winrate': wins / N_REPS})
    print(f"  ATE err: naive={np.mean(a_nv)!r}")
    print(f"           LDML ={np.mean(a_ld)!r}")
    print(f"           CF   ={np.mean(a_cf)!r}")
    print(f"  PEHE   : LDML median={np.median(p_ld)!r}")
    print(f"           CF   median={np.median(p_cf)!r}")
    print(f"           CF win rate={wins}/{N_REPS}")

S = pd.DataFrame(summary)
D = pd.DataFrame(per_real)
S.to_csv(f'{BASE}/ihdp_leaf_sweep.csv', index=False)
D.to_csv(f'{BASE}/ihdp_per_realisation.csv', index=False)

print("FULL GRID — report this table entire, in R10")
print(S.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

best = S.loc[S['PEHE_CF_median'].idxmin()]
print(f"\nBest CF PEHE at leaf = {int(best['min_samples_leaf'])} "
      f"(median PEHE {best['PEHE_CF_median']!r}, win rate {best['CF_PEHE_winrate']!r})")
if best['CF_PEHE_winrate'] > 0.5:
    print("  ==> PEHE recovers below leaf 10. The gap IS a leaf-size boundary condition.")
    print("      Report as: the engineered component is sample-size sensitive, here is")
    print("      the boundary, here is the configuration rule that follows.")
else:
    print("  ==> PEHE does NOT recover at any tested leaf size. The mis-scaling")
    print("      10/20/30 finding downward. Report as a characterised failure mode:")
    print("      a causal forest on a small, largely smooth response surface where a")
    print("      well-specified linear model is a strong competitor. That is a finding.")

print("PART B — NAIVE DIFFERENCE-IN-MEANS (the Rank 1 finding)")
r = S.iloc[0]
print(f"ATE recovery error, mean over {N_REPS} realisations (full precision):")
print(f"  naive difference-in-means : {r['ATEerr_naive_mean']!r}  +/- {r['ATEerr_naive_sd']!r}")
print(f"  Linear DML                : {r['ATEerr_LDML_mean']!r}  +/- {r['ATEerr_LDML_sd']!r}")
print(f"  CausalForestDML           : {r['ATEerr_CF_mean']!r}  +/- {r['ATEerr_CF_sd']!r}")

_nv = S['ATEerr_naive_mean'].round(12).nunique()
print(f"\nnaive ATE error constant across all {len(S)} leaf sizes? "
      f"{'YES' if _nv == 1 else f'NO -- {_nv} distinct values, LOOP IS LEAKING STATE'}")
print("\nPEHE for the naive estimator: NOT DEFINED. It produces no individual")
print("estimates at all -- which is the entire quantity this study is about.")
print("Two further points for R10, both available to you:")
print("  1. IHDP Setting B is semi-synthetic with known treated/control covariate")
print("     overlap, so difference-in-means here is not the naive estimator it would")
print("     be on observational field data. Its ATE recovery on this benchmark is not")
print("     evidence that adjustment is unnecessary in general.")

print("PART C — THE 20 OUTLIER REALISATIONS")
KNOWN = [8, 9, 12, 20, 25, 27, 33, 36, 38, 52, 59, 67, 70, 80, 81, 83, 84, 85, 92, 97]

d10 = D[D['leaf'] == 10]
if len(d10):
    thr = 3 * d10['pehe_cf'].median()
    found = sorted(d10.loc[d10['pehe_cf'] > thr, 'realisation'].tolist())
    print(f"threshold (3x median PEHE) = {thr!r}")
    print(f"identified: {found}")
    print(f"overlap: {len(set(found) & set(KNOWN))} of {len(KNOWN)}")

    out_mask = D['realisation'].isin(KNOWN)
    print("\nDoes the tail affect ALL estimators, or only the forest?")
    cmp = pd.DataFrame({
        'outlier realisations': [D.loc[out_mask, c].mean() for c in
                                 ['pehe_cf', 'pehe_ldml', 'ate_err_naive']],
        'other realisations':   [D.loc[~out_mask, c].mean() for c in
                                 ['pehe_cf', 'pehe_ldml', 'ate_err_naive']]},
        index=['CF PEHE', 'LDML PEHE', 'naive ATE error'])
    cmp['ratio'] = cmp['outlier realisations'] / cmp['other realisations']
    print(cmp.to_string(float_format=lambda v: f"{v:.9f}"))
    print("\nIf every estimator degrades on the same realisations, the tail is a")
    print("property of IHDP Setting B's exponential response surface -- which is what")
    print("If only CF degrades, it IS a characterisable failure mode of a causal forest")
    print("under a strongly non-linear surface at small n. Either answer is publishable.")

econml: 0.17.0 | sklearn: 1.6.1

PART A — LEAF SWEEP, EXTENDED BELOW 10

--- min_samples_leaf = 2 ---
    25/100  [169s]
    50/100  [337s]
    75/100  [507s]
    100/100  [676s]
  ATE err: naive=np.float64(0.28559182497003266)
           LDML =np.float64(0.5634733006467171)
           CF   =np.float64(0.6048059754220718)
  PEHE   : LDML median=np.float64(1.4300114273842028)
           CF   median=np.float64(1.6617529640308597)
           CF win rate=36/100

--- min_samples_leaf = 3 ---
    25/100  [843s]
    50/100  [1009s]
    75/100  [1175s]
    100/100  [1339s]
  ATE err: naive=np.float64(0.28559182497003266)
           LDML =np.float64(0.5634733006467171)
           CF   =np.float64(0.6026414457876189)
  PEHE   : LDML median=np.float64(1.4300114273842028)
           CF   median=np.float64(1.677274875551036)
           CF win rate=36/100

--- min_samples_leaf = 5 ---
    25/100  [1505s]
    50/100  [1671s]
    75/100  [1835s]
    100/100  [2000s]
  ATE err: naive=np.float64(0.28559